In [11]:
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun, ArxivQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

True

### Define LLM

In [5]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

### Tool Creation

In [ ]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    """Use this tool to search the web"""
    search = DuckDuckGoSearchRun()
    response =  search.invoke(query)
    return response


@tool
def tool_wikipedia_search(query: str) -> str:
    """Use this tool to get the results from Wikipedia"""
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    response = wikipedia.run(query)
    return response

@tool
def tool_arxiv_search(query: str) -> str:
    """Use this tool to get the research papers"""
    api_wrapper = ArxivAPIWrapper(
        doc_content_chars_max=2000,
        top_k_results=3
    )
    arxiv = ArxivQueryRun(api_wrapper=api_wrapper)
    response = arxiv.run(query)
    return response


#### Create a custom tool

In [20]:
@tool
def personl_info(query: str) -> str:
    """Use this tool when you need to answer questions about personal information"""
    infos = [
        {
            "name" : "John Doe",
            "age" : 30,
            "role": "Software Engineer"
        },
        {
            "name" : "Jane Smith",
            "age" : 28,
            "role": "Data Scientist"
        },
        {
            "name" : "Alice Johnson",
            "age" : 35,
            "role": "Product Manager"
        }
    ]
    
    for info in infos:
        if info['name'].lower() in query.lower():
            return f"Name: {info['name']}, Age: {info['age']}, Role: {info['role']}"
    return "No matching personal information found."


### Create Toolkit

In [21]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    personl_info
]

### Tool Binding

In [22]:
llm_bind = llm.bind_tools(toolkit)

In [ ]:
llm.invoke("What is the age of John Doe and what is his role?") ## random answer, as LLM does not know who is John Doe.

AIMessage(content='**John Doe** is not a specific, real person with a set age or role.\n\nInstead, "John Doe" (and "Jane Doe" for females) is a **placeholder name** used in various contexts when the true identity of an individual is unknown, needs to be kept anonymous, or is being used as a generic example.\n\n*   **Age:** Therefore, "John Doe" does not have a specific age. His age would be unknown or irrelevant, depending on the context in which the name is used.\n*   **Role:** His "role" varies widely depending on the situation:\n    *   **Legal/Police:** An unidentified defendant, plaintiff, victim, or body.\n    *   **Medical:** An unidentified patient.\n    *   **General Examples:** A generic user in software development, a placeholder name in a form, or an anonymous source.\n\nIf you are thinking of a specific fictional character named "John Doe" (for example, from a movie or book), please provide more context, and I might be able to tell you about that specific character.', addi

In [ ]:
llm_bind.invoke("What is the age of John Doe and what is his role?") ## LLM suggests to use the personl_info tool and gets the answer from it.

AIMessage(content='', additional_kwargs={'function_call': {'name': 'personl_info', 'arguments': '{"query": "John Doe age and role"}'}, '__gemini_function_call_thought_signatures__': {'37c1c924-aac7-486a-87fe-1a9e35461411': 'CqkCARFNMg/wT2bKWJyTmEED/mY0k2E8qVIJsr4Yc44GiIe6KPsOXzj9RXfpLQfey2Z7oU62QAwqWVhEnYgC9RdSSHE8OSLwBBNWO2iI5khg6nYqmACkFMC4dZjmtYjW6XAh0HnrZZ1JDnNOu2/Objc8suyHHODtR/LOziLcEFJWXXkPP82gvl55SjAe8sMhHFwbLz7MXJ/t7rZlvNrto1ra519YCraxTgTQO/40gMEZkfUAyQTW3puF1s6VQHUEKmF7NR2vmlIb2sJZeppYa6ov6RY+ExrusTqnBVLccADh52PDlgl3sxBD7cbI51l3oCFr7/UpDOeEBIOzab5AeNFGxwP20KDtZITY45WAUBmVliJQ4PLpsvbOZGfdy9jhRlAZ2AfRPF1u482E'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f55da-cb5b-7b10-9c2f-2f1131ecce23-0', tool_calls=[{'name': 'personl_info', 'args': {'query': 'John Doe age and role'}, 'id': '37c1c924-aac7-486a-87fe-1a9e35461411', 'type': 'tool_call'}], invalid_tool_calls=[], usage_meta

### Loop and Tool Execution using Agent

In [31]:
# An agent is a model calling tools in a loop until a given task is complete.

from langchain.agents import create_agent

my_agent = create_agent(llm_bind, toolkit)

In [33]:
my_agent.invoke(
    {"messages": [
        {"role": "user", "content": "What is the age of John Doe and what is his role?"}
        ]
    }
) 

{'messages': [HumanMessage(content='What is the age of John Doe and what is his role?', additional_kwargs={}, response_metadata={}, id='b74e1975-d3d3-478c-9173-2ac800fcbc18'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'personl_info', 'arguments': '{"query": "John Doe age and role"}'}, '__gemini_function_call_thought_signatures__': {'3ad35a35-e4e1-40dd-ad62-4776309116c3': 'CukCARFNMg+Lby7g/TUu7HDaiFE1iu12qROeHdd5YfrV/qe5REpAqubCPYuKmIWrR5/J8JqQ000WefagRQrJ44gDvWkXaEzbWpIPz6r8U16yAdihkx3fGrAHlsBS6g3qmjZ3CxNyY2m/FkEFoto/rcNB6W4FhSJtjOoNIVyy7UfgStDk0HmN6yag8FnNKa15yMptjdbT0ZAJuQ1WeMUWsYdRZjiQBC0L8A9ij4/XPnAVbtQUoDY4YOag9B4tItS8qhZT/r1YfFw5o/rtbiGpFCwHbDTplOJLk7VpBHh/UqQi0Rhx9HxhKVK88yhOEoR9mh8yigBrWRU5TMYTnkqhRoX8KIOkIXDQzKefYTVVIhP3HOx3BxleUY/2jMDc7eKeFcI9oEzUDfvFytNhnGAy+7z2YjiGi6vg5+NT174dgECnfrnBZ5YoG1PBkU/OtRfd5hl7jU+qU2oetceGpzMHzKTCATniD6cdidwf2g=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'mode